## Histogramming (4 bins: hB1, hB2, hB3, hB4)

In [1]:
%%writefile q7.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N 60
#define LOW 10
#define HIGH 99
#define BLOCK_SIZE 8

// ── CUDA Kernel
// Each thread atomically increments the correct bin
__global__ void histKernel(int *dA, int *dB1, int *dB2, int *dB3, int *dB4, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        int range = (HIGH - LOW + 1) / 4;   // = 22
        int bin   = (dA[i] - LOW) / range;
        if (bin > 3) bin = 3;
        if      (bin == 0) atomicAdd(dB1, 1);
        else if (bin == 1) atomicAdd(dB2, 1);
        else if (bin == 2) atomicAdd(dB3, 1);
        else               atomicAdd(dB4, 1);
    }
}

// ── Host Function
__host__ void histogram(int *hA, int *hB1, int *hB2, int *hB3, int *hB4, int n)
{
    int *dA, *dB1, *dB2, *dB3, *dB4;
    int size = n * sizeof(int);
    cudaMalloc((void**)&dA,  size);
    cudaMalloc((void**)&dB1, sizeof(int));
    cudaMalloc((void**)&dB2, sizeof(int));
    cudaMalloc((void**)&dB3, sizeof(int));
    cudaMalloc((void**)&dB4, sizeof(int));

    cudaMemcpy(dA, hA, size, cudaMemcpyHostToDevice);
    cudaMemset(dB1, 0, sizeof(int));
    cudaMemset(dB2, 0, sizeof(int));
    cudaMemset(dB3, 0, sizeof(int));
    cudaMemset(dB4, 0, sizeof(int));

    dim3 DimBlock(BLOCK_SIZE, 1, 1);
    dim3 DimGrid((int)ceil((float)n / BLOCK_SIZE), 1, 1);
    histKernel<<<DimGrid, DimBlock>>>(dA, dB1, dB2, dB3, dB4, n);

    cudaMemcpy(hB1, dB1, sizeof(int), cudaMemcpyDeviceToHost);
    cudaMemcpy(hB2, dB2, sizeof(int), cudaMemcpyDeviceToHost);
    cudaMemcpy(hB3, dB3, sizeof(int), cudaMemcpyDeviceToHost);
    cudaMemcpy(hB4, dB4, sizeof(int), cudaMemcpyDeviceToHost);

    cudaFree(dA); cudaFree(dB1); cudaFree(dB2); cudaFree(dB3); cudaFree(dB4);
}

// ── Main
int main()
{
    int *hA;
    int hB1 = 0, hB2 = 0, hB3 = 0, hB4 = 0;
    hA = (int*) malloc(N * sizeof(int));

    srand(time(NULL));
    for (int i = 0; i < N; i++)
        hA[i] = rand() % (HIGH - LOW + 1) + LOW;

    printf("hA:\n");
    for (int i = 0; i < N; i++) printf("%4d", hA[i]);
    printf("\n");

    histogram(hA, &hB1, &hB2, &hB3, &hB4, N);

    printf("\nHistogram bins (each covers 22 values):\n");
    printf("hB1 [10-31] = %d\n", hB1);
    printf("hB2 [32-53] = %d\n", hB2);
    printf("hB3 [54-75] = %d\n", hB3);
    printf("hB4 [76-99] = %d\n", hB4);
    printf("Total       = %d\n", hB1+hB2+hB3+hB4);

    free(hA);
    return 0;
}

Writing q7.cu


In [2]:
!nvcc -arch=sm_75 q7.cu -o q7

!./q7

hA:
  89  90  18  52  38  73  36  91  65  34  33  16  86  27  98  27  27  95  58  35  67  26  96  49  96  18  78  45  85  95  65  74  85  35  78  76  60  14  67  26  91  90  84  77  69  44  56  86  92  14  74  21  82  32  60  41  93  91  38  78

Histogram bins (each covers 22 values):
hB1 [10-31] = 11
hB2 [32-53] = 13
hB3 [54-75] = 12
hB4 [76-99] = 24
Total       = 60
